# Base Instruction Induction

In [ ]:
import numpy as np
import pandas as pd
import random
import ollama
from tqdm import tqdm
import re
import pickle
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error

In [ ]:
ollama_model = "llama3:70b-instruct"

# Building a 20-sample few-shot set for each Band Score

In [ ]:
raw_data = pd.read_excel('ielts.xlsx', engine='openpyxl')

In [ ]:
overall_indices = {
    score: raw_data.index[raw_data['Overall'] == score].tolist()
    for score in raw_data['Overall'].unique()
}

def shots_sampling(overall_indices, set_num):
    sample_list = []
    
    for _ in range(set_num):
        sample = []
        for score, indices in overall_indices.items():
            if len(indices) > 0:
                selected = random.sample(indices, 1)[0]
                sample.append(selected)
                indices.remove(selected)
        
        sample_list.append(sample)
    
    return sample_list

ii_list = shots_sampling(overall_indices, 20)

# 1st eval, 2nd Eval Index Sampling

In [ ]:
all_indices = list(range(len(raw_data)))
flat_used_indices = set(sum(ii_list, []))
remaining_indices = list(set(all_indices) - flat_used_indices)

In [ ]:
def random_sampling(remaining_indices, sample_size, sample_num):
    sampled_sets = []
    for _ in range(sample_num):
        sample = random.sample(remaining_indices, sample_size)
        sampled_sets.append(sample)
        remaining_indices = list(set(remaining_indices) - set(sample))
    return sampled_sets, remaining_indices

In [ ]:
sampled_sets, remaining_indices_after_sampling = random_sampling(remaining_indices, 100, 2)

# Instruction Induction

In [ ]:
# Prompt Format
iii3 = []
for i in ii_list:
    template = f'''I gave an IELTS Writing examiner an Instruction and Inputs that are Question-Answer Essay pairs. 
    The examiner read the instruction and scored every Input answer essay.
    
    Here are the Input Question-Answer pairs and Score:

    Question: {raw_data["Question"][i[0]]}
    Answer Essay: {raw_data["Essay"][i[0]]}
    Score: {raw_data["Overall"][i[0]]}

    Question: {raw_data["Question"][i[1]]}
    Answer Essay: {raw_data["Essay"][i[1]]}
    Score: {raw_data["Overall"][i[1]]}

    Question: {raw_data["Question"][i[2]]}
    Answer Essay: {raw_data["Essay"][i[2]]}
    Score: {raw_data["Overall"][i[2]]}

    Question: {raw_data["Question"][i[3]]}
    Answer Essay: {raw_data["Essay"][i[3]]}
    Score: {raw_data["Overall"][i[3]]}

    Question: {raw_data["Question"][i[4]]}
    Answer Essay: {raw_data["Essay"][i[4]]}
    Score: {raw_data["Overall"][i[4]]}

    Based on these Input pairs and Scores, please infer the instruction that was given to the examiner. Infer instructions without further explanation.'''
    
    iii3.append(template)

In [ ]:
induction = []
for prompt in iii3:
    
    response = ollama.chat(
        model=ollama_model,
        messages=[
        {"role": "system", "content": "You are a prompt Engineer. Generate the instruction without further explanation."},
        {'role': 'user', 'content': prompt}],
        options={"num_keep": 100,
        },
        )
    print(response['message']['content'])
    induction.append(response['message']['content'])

In [ ]:
induct = pd.DataFrame({"induction_set" : induction})
induct.to_csv('inductions.csv', encoding='utf-8-sig', index=False)

# First Evaluation

In [ ]:
instruction_cleaned = pd.read_csv('inductions.csv')
instruction_list = list(instruction_cleaned['induction_set'])
test = pd.read_excel('evaluation_1.xlsx', engine='openpyxl')

In [ ]:
test_q = list(test["Question"])
test_a = list(test["Essay"])
test_output = list(test["Overall"])

In [ ]:
# Prompt Format
prompt_dic = {}
number = 0

for instruction in instruction_list:
    input_prompt = []
    
    for question, answer in zip(test_q, test_a):
        prompt = f"""{instruction}

Question: {question}
Answer Essay: {answer}
Score: """
        input_prompt.append(prompt)
    
    prompt_dic[number] = input_prompt
    number += 1

In [ ]:
# Start evalution
result = {}

for i in tqdm(range(len(instruction_list))):
    
    prompt_list = prompt_dic[i]
    result_list = []
    
    for each_prompt in prompt_list:
        response = ollama.chat(
            model=ollama_model,
            messages=[
                {"role": "system", "content": "You are a helpful chatbot of this scoring task. Generate only an overall band score without further explanation. Generate the band score as an integer."},
                {'role': 'user', 'content': each_prompt}],
            )
        print(response['message']['content'])
        result_list.append(response['message']['content'])
    
    result[i] = result_list

In [ ]:
result_df = pd.DataFrame.from_dict(result, orient='index')
result_df.to_excel('firstresult.xlsx')

# Scoring

In [ ]:
test_c = pd.read_excel('evaluation_1.xlsx', engine='openpyxl')
test_e = pd.read_excel('firstresult.xlsx', engine='openpyxl')

In [ ]:
test_c_list = test_c['Overall'].tolist()
test_e_lists = test_e.values.tolist()

In [ ]:
# QWK
qwk_list = []

from sklearn.metrics import cohen_kappa_score

for clean_test_e_list in test_e_lists:
    kappa_quadratic = cohen_kappa_score(test_c_list, clean_test_e_list, weights='quadratic', labels=[5,6,7,8,9])
    qwk_list.append(kappa_quadratic)
# IELTS labels=[5,6,7,8,9], ASAP++ labels=[0,1,2,3], ELLIPSE labels=[1,2,3,4,5]

In [ ]:
# Executive Accuracy 
score_dict = {}
score_list = []

for i, clean_test_e_list in enumerate(test_e_lists):
    diff = [pred == real for pred, real in zip(clean_test_e_list, test_c_list)]
    score_list.append(sum(diff))
    score_dict[i] = diff

In [ ]:
# mse
mse_scores = []

for clean_test_e_list in test_e_lists:
    mse = mean_squared_error(test_c_list, clean_test_e_list)
    mse_scores.append(mse)

In [ ]:
# rmse
rmse_scores = []

for clean_test_e_list in test_e_lists:
    mse = mean_squared_error(test_c_list, clean_test_e_list)
    rmse = np.sqrt(mse)
    rmse_scores.append(rmse)